# Hope Archive · 常用命令手册

这个 Jupyter Notebook 用于学习和操作现有 Python CLI，不替代项目代码。

**当前流程：Diary Downloader → Normalization → Media Localization → Markdown Export。**
PDF 由你使用自己的工具导出，本项目不自动转换。

## 如何打开

在 VS Code 中打开本文件，选择 Python kernel。推荐选择项目 `.venv`。
如果提示缺少 kernel，在项目根目录 PowerShell 中执行：

```powershell
.\.venv\Scripts\python.exe -m pip install ipykernel
```

这只用于 Notebook；原有 CLI 不依赖 Jupyter。`Shift+Enter` 运行当前 cell。
Notebook 中的操作命令默认只预览；需要执行时，将对应 cell 的 `execute=False` 改成 `execute=True`。
因此直接 Run All 不会下载日记、下载媒体或生成文件。

运行后若出现私人信息，分享 Notebook 前使用 **Clear All Outputs** 并检查内容。


In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT = Path(r"D:/AUniversityLearning/3102/CODING/HopeProject")
PYTHON = PROJECT / ".venv" / "Scripts" / "python.exe"
assert PROJECT.is_dir(), f"Project not found: {PROJECT}"
assert PYTHON.is_file(), f"Python not found: {PYTHON}"

def run(arguments, execute=False):
    """Use the project interpreter and argument lists; no shell quoting required."""
    command = [str(PYTHON), "-B", *map(str, arguments)]
    print(subprocess.list2cmdline(command))
    if execute:
        result = subprocess.run(command, cwd=PROJECT, check=False)
        print("Exit code:", result.returncode)
        if result.returncode != 0:
            raise RuntimeError("Command failed. Review the output above before continuing.")
    else:
        print("Preview only. Set execute=True in this cell to run.")

print("Project:", PROJECT)
print("Python:", PYTHON)


## 1. 查看 CLI 帮助

`--help` 显示参数，不会发送 API request。即使 Notebook kernel 来自别的环境，下面的命令也使用项目 `.venv`。


In [ ]:
for script in ["main.py", "normalize.py", "media.py", "export_markdown.py"]:
    run(["src/hope_archive/" + script, "--help"], execute=False)


## 2. 使用已有数据

以下默认路径对应目前已有的日记和媒体。只想重新生成 Markdown 时，直接使用第 6 节即可，不必重新下载。


In [ ]:
DOWNLOADER_INPUT = PROJECT / "data/processed/diaries.json"
NORMALIZED_OUTPUT = PROJECT / "data/processed/diaries.normalized.json"
ARCHIVE = PROJECT / "data/archive"

for path in [DOWNLOADER_INPUT, NORMALIZED_OUTPUT, ARCHIVE / "media_manifest.json"]:
    print(path.relative_to(PROJECT), "exists:", path.exists())


## 3. 完整归档（可选，访问网络）

main.py 与 UI 共用 application.export_archive()，自动下载、标准化、保存媒体并生成 Markdown。每次在 OUTPUT_ROOT 中创建独立子目录；无须手动接续各阶段。userId 临时输入，不写入 cell。


In [ ]:
from getpass import getpass

BEGIN_DATE = "2026-09-01"
END_DATE = "2026-09-08"
OUTPUT_ROOT = PROJECT / "data/run_002"
EXECUTE_DOWNLOAD = False

arguments = [
    "src/hope_archive/main.py",
    "--begin-date", BEGIN_DATE, "--end-date", END_DATE,
    "--note-type", "0",
    "--output-dir", str(OUTPUT_ROOT),
]
if EXECUTE_DOWNLOAD:
    user_id = getpass("Your own backend userId: ")
    if not user_id.strip():
        raise ValueError("userId must not be empty")
    try:
        # Avoid printing the actual ID in Notebook output.
        result = subprocess.run(
            [str(PYTHON), "-B", *arguments, "--user-id", user_id],
            cwd=PROJECT, check=False,
        )
        print("Exit code:", result.returncode)
        if result.returncode != 0:
            raise RuntimeError("Archive failed; inspect the terminal output.")
    finally:
        del user_id
else:
    run([*arguments, "--user-id", "YOUR_BACKEND_USER_ID"])


### 选择已完成的归档进行检查

完整归档不必重复 normalization。把终端显示的实际 Output 路径填入 RUN_DIR；第 5/6 节仅用于重试媒体或重新导出，第 7 节检查数量。第 4 节仅适合还没有 normalized JSON 的旧数据。


In [ ]:
USE_NEW_BATCH = False
if USE_NEW_BATCH:
    RUN_DIR = Path("REPLACE_WITH_ACTUAL_OUTPUT_DIRECTORY")
    DOWNLOADER_INPUT = RUN_DIR / "processed/diaries.json"
    NORMALIZED_OUTPUT = RUN_DIR / "processed/diaries.normalized.json"
    ARCHIVE = RUN_DIR / "archive"
    assert NORMALIZED_OUTPUT.is_file(), "Select a completed archive run"
    print("Selected archive:", RUN_DIR)


## 4. Normalization（本地转换）

读取 combined JSON，保存新的 normalized JSON，不修改 raw JSON。
默认 normalized output 已存在时会拒绝覆盖；此时无需重跑，直接继续第 5/6 节。


In [ ]:
run([
    "src/hope_archive/normalize.py",
    "--input", DOWNLOADER_INPUT,
    "--output", NORMALIZED_OUTPUT,
], execute=False)


## 5. Media Localization（下载媒体）

下载远程原始 bytes，并保存 `media_manifest.json`。
已有媒体通过 size/hash 校验后 skip；单个失败会记录在 manifest，可再次运行 retry。
本节可能访问网络；只想导出已有媒体时不需要重跑。


In [ ]:
run([
    "src/hope_archive/media.py",
    "--input", NORMALIZED_OUTPUT,
    "--archive", ARCHIVE,
], execute=False)


## 6. Markdown Export（本地导出）

每篇日记一个 `.md`，图片引用 archive 中的本地 relative path。
相同内容重跑会 skip；不同内容不会覆盖已有 Markdown。
没有本地媒体时会保留 unavailable 提示，不会自动下载。


In [ ]:
run([
    "src/hope_archive/export_markdown.py",
    "--input", NORMALIZED_OUTPUT,
    "--archive", ARCHIVE,
], execute=False)


## 7. 检查输出数量（只读）

只显示统计，不打印日记正文。Markdown 数量相同不独立证明内容完全正确，仍需人工抽查。
manifest status 是最近一次执行状态：`downloaded` 与 `skipped` 都代表当时成功获得的媒体。


In [ ]:
from collections import Counter

if NORMALIZED_OUTPUT.exists():
    document = json.loads(NORMALIZED_OUTPUT.read_text(encoding="utf-8"))
    diary_count = len(document["diaries"])
    markdown_count = len(list(ARCHIVE.rglob("*.md"))) if ARCHIVE.exists() else 0
    print("Diary entries:", diary_count)
    print("Markdown files:", markdown_count)
    print("Counts match:", diary_count == markdown_count)
else:
    print("Normalized JSON does not exist yet.")

manifest_path = ARCHIVE / "media_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("Media status:", dict(Counter(record.get("status") for record in manifest["media"].values())))


## 8. 运行 automated tests

使用合成数据和临时目录，不访问 Hope，也不修改真实归档。遇到预期错误日志时看最后的 `OK` / `FAILED`，不要仅根据中间的 “failed” 字样判断测试失败。


In [ ]:
run(["-m", "unittest", "discover", "-s", "tests", "-v"], execute=False)


## 9. 查看 Git 状态（只读）

不要提交真实 diary JSON、媒体、私人配置或带私人输出的 Notebook。


In [ ]:
EXECUTE_GIT_CHECK = False
if EXECUTE_GIT_CHECK:
    for arguments in [["git", "status", "--short"], ["git", "diff", "--stat"], ["git", "diff", "--check"]]:
        subprocess.run(arguments, cwd=PROJECT, check=True)
else:
    print("Preview: git status --short; git diff --stat; git diff --check")


## 10. PowerShell 速查

以下是终端命令，不是 Python cell。请复制到 PowerShell：

```powershell
Set-Location 'D:/AUniversityLearning/3102/CODING/HopeProject'
.\.venv\Scripts\python.exe --version
.\.venv\Scripts\python.exe -B src\hope_archive\export_markdown.py --help
.\.venv\Scripts\python.exe -B src\hope_archive\export_markdown.py
.\.venv\Scripts\python.exe -B -m unittest discover -s tests -v
Get-ChildItem data/archive -Recurse -Filter *.md | Measure-Object
git status --short
git diff --stat
git diff --check
```

## 必须理解的概念

- **kernel**：执行 Notebook cell 的 Python 进程。
- **virtual environment**：项目独立 Python 环境；这里明确使用 `.venv` 的解释器执行 CLI。
- **subprocess**：从 Python 调用已有命令，不重复实现业务逻辑。
- **Path**：安全组合 Windows 文件路径。
- **cell state**：变量保存在 kernel 内存中；重启 kernel 后先重新运行配置 cell。
- **relative path**：Markdown 图片链接依赖整个 archive 的目录关系；搬动时一起搬媒体。

本 Notebook 不含删除、覆盖、自动 PDF 转换或向服务器写入的命令。
